# 01 — Data Collection

**Goal:** assemble the unified review table. The pipeline is source-agnostic (config switch `scrape` | `fallback`); here we use the ToS-safe McAuley/UCSD **Amazon Reviews 2023** fallback, auto-detecting whatever categories are present in `data/raw/`. Each loader emits the same 16-column unified schema, with a product-metadata **join** (multi-source).

> **Weak-proxy caveat.** Label = Amazon *Verified Purchase* (purchase verification, **not** ground-truth deception). Strengthened with behavioral signals; validated against real fake-review labels in notebook 07.

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from src import utils, viz
config = utils.load_config()
utils.set_seeds(config["project"]["random_seed"])
viz.set_house_style()
FIG = ROOT / "reports" / "figures"; FIG.mkdir(parents=True, exist_ok=True)

In [2]:
from src import fallback_loader as fl, clean
raw = ROOT / "data" / "raw"
cats = sorted({p.name[:-9] for p in raw.glob("*.jsonl.gz") if not p.name.startswith("meta_")})
assert cats, "No data in data/raw/. Download a category (see notebook 01 / README)."
print("categories present:", cats)
df = clean.clean_reviews(pd.concat(
    [pd.DataFrame(fl.load_category(config, c, raw_dir=str(raw))) for c in cats], ignore_index=True))
print("reviews:", len(df), "| verified rate:", round((df["verified_purchase"] == True).mean(), 3))

categories present: ['Subscription_Boxes']
2026-06-09 02:56:54,239 | review_deception.fallback | INFO | Loaded 16216 reviews for category 'Subscription_Boxes' from Subscription_Boxes.jsonl.gz


2026-06-09 02:56:54,395 | review_deception.clean | INFO | Cleaned 16216 reviews (447 flagged exact-duplicate).


reviews: 16216 | verified rate: 0.88


In [3]:
print(df[["review_id","product_id","reviewer_id","rating","verified_purchase","helpful_votes","product_title"]].head().to_string(index=False))
print("\nunified columns:", list(df.columns))
print("products:", df["product_id"].nunique(), "| reviewers:", df["reviewer_id"].nunique())
print("date range:", df["review_date"].min().date(), "->", df["review_date"].max().date())
print("verified:", int((df["verified_purchase"]==True).sum()), "| unverified:", int((df["verified_purchase"]==False).sum()))

       review_id product_id                  reviewer_id  rating  verified_purchase  helpful_votes                                                                                                                                  product_title
b13e0ec80fe499b9 B09WC47S3V AEMJ2EG5ODOCYUTI54NBXZHDJGSQ     1.0               True              2 KitNipBox | Happy Cat Box | Monthly Cat Subscription Boxes Filled with Cat Toys, Kitten Toys, North American Grown Catnip Toys, and Cat Treats
1bfbdb93f00ab44c B07QL1JRCN AEEJBFZKUBEEMBZUZJV4UHFVEEBQ     2.0               True             20                                                                                               Lip Monthly - Beauty and Makeup Subscription Box
14a4f97151624c8a B08N5QKX1Y AGSVZNZBTSGQBKZDZTQYEZHGDPCQ     1.0               True              4      BarkBox Monthly Subscription Box, Dog Chew Toys, All Natural Dog Treats, Dental Chews, Dog Supplies Themed Monthly Box, Large Dog (50lb+)
e9441b52f9e28e59 B07KM6T8GV AFDE

**Collection engineering (see `src/scrape.py`).** The live scraper is polite by design (rate limiting, exponential backoff, UA rotation, gzipped cache, checkpoint/resume) and degrades gracefully to this dataset. Live scraping is gated; run the offline demo with `python -m src.scrape --dry-run`. The dataset above is the documented, reproducible path.